<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/02_knowledge_graph_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install requests

## Setting up Environment


In [2]:
import json
import re
from pathlib import Path
import requests
print("Environment Ready")

Environment Ready


## Loading chunks from Notebook 1

In [5]:
from pathlib import Path
import json

# Automatically mount Google Drive if a Drive path is requested
drive_path = Path("/content/drive/MyDrive/HybridGraphRAG/document_chunks.jsonl")
local_path = Path("/content/processed/document_chunks.jsonl")

# Check and mount if needed
if not drive_path.exists():
    try:
        from google.colab import drive
        print("Google Drive path not found. Attempting to mount Google Drive...")
        drive.mount('/content/drive')
    except ImportError:
        pass

# Determine which path to use
if drive_path.exists():
    chunks_path = drive_path
    print(f"Using Google Drive path: {chunks_path}")
elif local_path.exists():
    chunks_path = local_path
    print(f"Using local session path: {chunks_path}")
else:
    raise FileNotFoundError(
        f"Could not find the document_chunks.jsonl file at either:\n"
        f"- {drive_path}\n"
        f"- {local_path}\n"
        "Please ensure you have mounted your Google Drive or run Notebook 1 in this session."
    )

chunks = []
with chunks_path.open("r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print("Number of chunks: ", len(chunks))

Google Drive path not found. Attempting to mount Google Drive...
Mounted at /content/drive
Using Google Drive path: /content/drive/MyDrive/HybridGraphRAG/document_chunks.jsonl
Number of chunks:  102


In [7]:
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

{
  "id": "12396518_p1_c0",
  "text": "Published online 23 August 2002\nBehavioural models of population growth rates:\nimplications for conservation and prediction\nWilliam J. Sutherland1* and Ken Norris2\n1Centre for Ecology, Evolution and Conservation, University of East Anglia, Norwich NR4 7TJ, UK\n2School of Animal & Microbial Sciences, University of Reading, Whiteknights, PO Box 228, Reading, RG6 6AJ, UK\nConservation biologists often wish to predict how vertebrate populations will respond to local or global\nchanges in conditions such as those resulting from sea-level rise, deforestation, exploitation, genetically\nmodiﬁed crops, global warming, human disturbance or from conservation activities. Predicting the conse-\nquences of such changes almost always requires understanding the population growth rate and the",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0
}


## Extraction Pipeline

In [10]:
EXTRACTION_SCHEMA = {
    "entities": [
        {
            "name": "string",
            "type": "string"
        }
    ],
    "relationships": [
        {
            "source": "string",
            "relation": "string",
            "target": "string"
        }
    ]
}
print(json.dumps(EXTRACTION_SCHEMA, indent=2))

{
  "entities": [
    {
      "name": "string",
      "type": "string"
    }
  ],
  "relationships": [
    {
      "source": "string",
      "relation": "string",
      "target": "string"
    }
  ]
}


## Extraction Prompt

In [11]:
def build_extraction_prompt(text):
    return f"""
You are an information extraction system.

Extract important entities and relationships from the text below.

ENTITY TYPES may include:
- PERSON
- ORGANIZATION
- MODEL
- METHOD
- DATASET
- CONCEPT
- TECHNOLOGY
- TASK
- METRIC
- PAPER
- OTHER

For each relationship:
- source must be an entity name
- target must be an entity name
- relation should be a short uppercase phrase such as:
  USES
  BASED_ON
  DEVELOPED_BY
  EVALUATED_ON
  IMPROVES
  USED_FOR
  PART_OF
  COMPARED_WITH

Rules:
1. Only extract information explicitly supported by the text.
2. Do not invent entities.
3. Do not invent relationships.
4. Keep entity names concise.
5. Return valid JSON only.
6. Do not include markdown code fences.

Required format:

{{
  "entities": [
    {{
      "name": "Entity name",
      "type": "ENTITY_TYPE"
    }}
  ],
  "relationships": [
    {{
      "source": "Entity A",
      "relation": "RELATION",
      "target": "Entity B"
    }}
  ]
}}

TEXT:
{text}
"""

In [13]:
test_prompt = build_extraction_prompt(chunks[0]["text"])
print(test_prompt)


You are an information extraction system.

Extract important entities and relationships from the text below.

ENTITY TYPES may include:
- PERSON
- ORGANIZATION
- MODEL
- METHOD
- DATASET
- CONCEPT
- TECHNOLOGY
- TASK
- METRIC
- PAPER
- OTHER

For each relationship:
- source must be an entity name
- target must be an entity name
- relation should be a short uppercase phrase such as:
  USES
  BASED_ON
  DEVELOPED_BY
  EVALUATED_ON
  IMPROVES
  USED_FOR
  PART_OF
  COMPARED_WITH

Rules:
1. Only extract information explicitly supported by the text.
2. Do not invent entities.
3. Do not invent relationships.
4. Keep entity names concise.
5. Return valid JSON only.
6. Do not include markdown code fences.

Required format:

{
  "entities": [
    {
      "name": "Entity name",
      "type": "ENTITY_TYPE"
    }
  ],
  "relationships": [
    {
      "source": "Entity A",
      "relation": "RELATION",
      "target": "Entity B"
    }
  ]
}

TEXT:
Published online 23 August 2002
Behavioural models

## Mock extractor

In [14]:
def mock_extract(text):
  return{
      "entities":[],
      "relationships":[]
  }

In [15]:
result = mock_extract(chunks[0]["text"])
print(json.dumps(result, indent=2))

{
  "entities": [],
  "relationships": []
}


## Building a validator

In [16]:
from typing_extensions import Required
def validate_extraction(data):
  if not isinstance(data, dict):
    return False

  if "entities" not in data:
    return False

  if "relationships" not in data:
    return False

  if not isinstance(data["entities"], list):
    return False

  if not isinstance(data["relationships"], list):
    return False

  for entity in data["entities"]:
    if not isinstance(entity, data):
      return False

    if "name" not in entity or "type" not in entity:
      return False

  for relationship in data["relationships"]:
    if not isinstance(relationship, dict):
      return False

    required = {"source", "relation", "target"}

    if not required.issubset(relationship.keys()):
      return False

    if not required.issubset(relationship.keys()):
      return False

  return True



In [17]:
print(validate_extraction(result))

True


## Normalise

In [18]:
def normalize_entity_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return name

In [22]:
examples = [
    "   BERT  ",
    "BERT  model",
    "  Transformer   "
]

for item in examples:
    print(repr(normalize_entity_name(item)))

'BERT'
'BERT model'
'Transformer'


## A cleaner function

In [23]:
def normalize_extraction(data):
  entities = []
  relationships = []
  seen_entities = set()

  for entity in data.get("entities", []):
    name = normalize_entity_name(entity["name"])
    entity_type = str(entity["type"]).strip().upper()

    key = (name.lower(), entity_type)

    if key not in seen_entities:
      entities.append({
          "name": name,
          "type": entity_type
      })
      seen_entities.add(key)

    for rel in data.get("relationships", []):
      source = normalize_entity_name(rel["source"])
      relation = str(rel["relation"]).strip().upper()
      target = normalize_entity_name(rel["target"])

      if source and target and relation:
        relationships.append({
            "source": source,
            "relation": relation,
            "target": target
        })

  return {
      "entities": entities,
      "relationships": relationships
  }


## Creating a graph record for each count

In [24]:
def create_graph_record(chunk, extraction):
    return {
        "chunk_id": chunk["id"],
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_index": chunk["chunk_index"],
        "entities": extraction["entities"],
        "relationships": extraction["relationships"]
    }

In [25]:
normalized = normalize_extraction(result)

graph_record = create_graph_record(
    chunks[0], normalized
)
print(json.dumps(graph_record, indent=2))

{
  "chunk_id": "12396518_p1_c0",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0,
  "entities": [],
  "relationships": []
}


## outpur dir

In [26]:
output_dir = Path("/content/processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [28]:
test_records = [graph_record]

graph_output = output_dir / "graph_records.jsonl"

with graph_output.open("w", encoding="utf-8") as f:
    for record in test_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("Saved:", graph_output)

Saved: /content/processed/graph_records.jsonl


In [29]:
with graph_output.open("r", encoding="utf-8") as f:
    loaded_graph_records = [
        json.loads(line)
        for line in f
    ]

print(json.dumps(
    loaded_graph_records[0],
    indent=2,
    ensure_ascii=False
))

{
  "chunk_id": "12396518_p1_c0",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0,
  "entities": [],
  "relationships": []
}
